In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.decomposition import PCA

In [2]:
root_path = "/home/stefan/ioai-prep/kits/pre-iaio/meaning-of-life"
seed = 42

# Theory Background

## Subspace Representation

A $k$-dimensional subspace $S$ in $\mathbb{R}^{D}$ (where $D=10, k=3$) can be defined by a mean vector $\mathbf{\mu} \in \mathbb{R}^{D}$ and an orthonormal basis $\mathbf{V} = [\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_k]$. Any point $\mathbf{x} \in S$ can be expressed as:

$$ \mathbf{x} = \mathbf{\mu} + \sum_{j=1}^{k} a_j \mathbf{v}_j = \mathbf{\mu} + \mathbf{V}\mathbf{a} $$

where $\mathbf{a} \in \mathbb{R}^{k}$ is the vector of coordinates in the subspace.

# Data

In [3]:
df = pd.read_csv(f"{root_path}/train_data.csv")
df.head()

,subtaskID,datapointID,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10
0,2,1,18.845863,15.432435,-4.888494,-11.196174,1.408862,-17.684394,3.231677,-1.476026,-4.660365,-15.947031
1,2,2,-4.746868,-5.971201,4.346016,-6.677494,1.910919,7.616225,-0.325792,-8.837899,-10.356530,-4.331975
2,2,3,2.908971,-19.955318,-11.212437,9.294100,-1.321366,-5.237313,-13.585740,-10.797363,-20.217986,-1.012807
3,2,4,-12.374266,1.094085,13.286408,3.131845,-6.065034,4.559042,-4.590903,-6.946004,-11.543627,-17.518288
4,2,5,3.261330,-12.511136,9.240270,-1.849021,-5.227230,10.490092,-7.043437,-14.084613,-15.566292,6.060100


# Subtask 1

In [4]:
# is 51 prime?
# used a few quantum computers for calculations
subtask1 = 0

# Subtask 2: PCA method

PCA via SVD on the centered data matrix $X - \mu$ yields $X - \mu = U\Sigma V^T$. The top $k=3$ right singular vectors form the basis $V_k$ of the principal subspace, which captures maximum variance.

The orthogonal projection of a point onto the subspace is:

$$ \hat{\mathbf{x}} = \mathbf{\mu} + V_k^T V_k (\mathbf{x} - \mathbf{\mu}) $$

The reconstruction error equals the distance to the subspace:

$$ d(\mathbf{x}, S) = \|\mathbf{x} - \hat{\mathbf{x}}\| $$

Points with small reconstruction error are candidates for lying on $S$. We take the 20 points with lowest error.

**Why it underperforms:** PCA is fit on all 420 points. With 400 noise points dominating, the recovered subspace is biased toward the noise distribution, not the true 3D subspace.

In [5]:
X = df[[f"X{i}" for i in range(1, 11)]]

In [6]:
pca = PCA(n_components=3)
X_reduced = pca.fit_transform(X)
X_reconstructed = pca.inverse_transform(X_reduced)

In [ ]:
errors = np.linalg.norm(X - X_reconstructed, axis=1)

pca_indices = np.argpartition(errors, 20 - 1)[:20]

pca_indices

array([ 5.79856874,  9.04636031,  9.05738678,  9.95952002,  9.12484429,
        9.25059309, 10.2579416 , 10.49509274, 11.01790895, 11.24310271,
       11.43670327, 11.42855795, 10.61417201, 10.91754058, 11.65726103,
       11.70044535, 12.033086  , 12.14869408, 12.30880985, 12.43615198])

Scores 0.45 F1, 14 points.

# Subtask 2 - ransac-like method

Standard PCA is sensitive to outliers. Since only 20 out of 420 points are inliers, we use a Random Sample Consensus (RANSAC) approach to find the optimal subspace $S$.

### 1. Minimal Sample and Affine Subspace
A $k$-dimensional affine subspace is defined by $k+1$ points. In this task, $k=3$, so we sample 4 points. Let these points be $\{\mathbf{x}_1, \dots, \mathbf{x}_4\}$. We define the local mean:
$$ \mathbf{\mu}_{sample} = \frac{1}{4} \sum_{i=1}^4 \mathbf{x}_i $$
Centering the sample points gives the matrix $\mathbf{X}_{centered} \in \mathbb{R}^{4 \times 10}$.

### 2. SVD and Basis Extraction
To find the best-fitting 3D subspace for this sample, we use Singular Value Decomposition (SVD):
$$ \mathbf{X}_{centered} = \mathbf{U}\mathbf{\Sigma}\mathbf{V}^T $$
The first $k=3$ columns of $\mathbf{V}$ (or rows of $\mathbf{V}^T$) provide the orthonormal basis $\mathbf{B} = [\mathbf{v}_1, \mathbf{v}_2, \mathbf{v}_3]$. This basis spans the directions of highest variance within the sample.

### 3. Orthogonal Projection and Distance
The distance from any point $\mathbf{x}$ to the subspace $S$ is the norm of the vector component orthogonal to $S$. We compute this using the projection matrix $\mathbf{P} = \mathbf{B}\mathbf{B}^T$:
$$ \text{proj}_S(\mathbf{x} - \mathbf{\mu}) = \mathbf{B}\mathbf{B}^T(\mathbf{x} - \mathbf{\mu}) $$
Using the Pythagorean theorem, the squared Euclidean distance is:
$$ d(\mathbf{x}, S)^2 = \|\mathbf{x} - \mathbf{\mu}\|^2 - \|\mathbf{B}^T(\mathbf{x} - \mathbf{\mu})\|^2 $$
The algorithm uses this formula to score the entire dataset against the candidate subspace.

### 4. Consensus and Refinement
The "All-Knowing" points are defined as the 20 points minimizing $d(\mathbf{x}, S)$. 

**Refinement (Total Least Squares):**
A minimal sample of 4 points is highly sensitive to noise. Once the 20 candidate inliers are identified, we perform a "Refit":
1.  Re-calculate $\mathbf{\mu}$ and $\mathbf{B}$ using all 20 candidate points.
2.  This is equivalent to finding the Total Least Squares solution for the subspace, which minimizes:
    $$ \min_{\mathbf{\mu}, \mathbf{B}} \sum_{i \in \text{Inliers}} \|\mathbf{x}_i - (\mathbf{\mu} + \mathbf{B}\mathbf{B}^T(\mathbf{x}_i - \mathbf{\mu}))\|^2 $$
This refinement step is critical for convergence to the true subspace $S$ amidst the noise $\gamma$.

In [32]:
X_np = X.values
N, D = X_np.shape
n_inliers = 20
n_dims_subspace = 3

iterations = 200000

best_error = float("inf")
best_indices = None

In [33]:
for i in tqdm(range(iterations)):
    # sample 4 points
    sample_idx = np.random.choice(N, n_dims_subspace + 1, replace=False)
    sample_pts = X_np[sample_idx]

    # create subspace
    mean = np.mean(sample_pts, axis=0)
    centered_sample = sample_pts - mean
    _, _, Vt = np.linalg.svd(centered_sample, full_matrices=False)
    basis = Vt[:n_dims_subspace]

    # distance as error
    X_centered = X_np - mean
    norms_X = np.sum(X_centered**2, axis=1)
    norms_proj = np.sum((X_centered @ basis.T) ** 2, axis=1)
    errors = np.sqrt(np.maximum(0, norms_X - norms_proj))

    # partial sort for speed
    top_20_idx = np.argpartition(errors, n_inliers - 1)[:n_inliers]
    total_error = np.sum(errors[top_20_idx])
    
    # refinement step (refit)
    if total_error < best_error:
        refine_pts = X_np[top_20_idx]
        refine_mean = np.mean(refine_pts, axis=0)
        refine_centered = refine_pts - refine_mean

        _, _, refine_Vt = np.linalg.svd(refine_centered, full_matrices=False)
        refine_basis = refine_Vt[:n_dims_subspace]

        refine_X_centered = X_np - refine_mean
        refine_norms_X = np.sum(refine_X_centered**2, axis=1)
        refine_norms_proj = np.sum((refine_X_centered @ refine_basis.T) ** 2, axis=1)
        refine_errors = np.sqrt(np.maximum(0, refine_norms_X - refine_norms_proj))

        refine_top_20_idx = np.argpartition(refine_errors, n_inliers - 1)[:n_inliers]
        refine_total_error = np.sum(refine_errors[refine_top_20_idx])

        if refine_total_error < best_error:
            best_error = refine_total_error
            best_indices = refine_top_20_idx.copy()
            print(f"Iteration {i} - New best error: {best_error:.4f}")

  1%|          | 1580/200000 [00:00<00:24, 8002.69it/s]

Iteration 0 - New best error: 219.8261
Iteration 1 - New best error: 185.7646
Iteration 11 - New best error: 170.6702
Iteration 133 - New best error: 156.3429
Iteration 768 - New best error: 139.7255


  3%|▎         | 6920/200000 [00:00<00:22, 8524.45it/s]

Iteration 5583 - New best error: 52.3095


100%|██████████| 200000/200000 [00:23<00:00, 8542.84it/s]


In [34]:
best_indices

array([  1, 265, 399, 348,  78, 239, 357, 387, 381, 267, 414, 354, 124,
       169, 273, 134, 310,  34, 305, 276])

In [35]:
subtask2 = np.zeros((X.shape[0],), dtype=int)
subtask2[best_indices] = 1

Scores ~70 points without the refinement step, 99 points with refinement.

# Submission

In [36]:
def build_subtask(ans, sid):
    return pd.DataFrame({"subtaskID": sid, "datapointID": df["datapointID"] if sid == 2 else [1], "answer": ans})


subtasks = [
    (subtask1, 1),
    (subtask2, 2),
]

submission_df = pd.concat([build_subtask(ans, sid) for ans, sid in subtasks])

In [37]:
submission_df.head()

,subtaskID,datapointID,answer
0,1,1,0
0,2,1,0
1,2,2,1
2,2,3,0
3,2,4,0


In [38]:
submission_df.to_csv(f"{root_path}/submission.csv", index=False)